# 07 — Advanced GINE + frozen ESM-2 + cross-attention

## Goal

Precompute frozen residue-level ESM-2 embeddings and configure one advanced experiment: four residual GINE layers, a three-layer length-preserving protein CNN, bidirectional masked cross-attention, masked pooling, and a `256 → 512 → 256 → 1` regressor.

**Execution policy:** cache generation, smoke tests, and training are independently disabled by default.


## Setup


In [ ]:
from pathlib import Path
from importlib import import_module
import json
import subprocess
import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project checkout.")


def require_api(module_name: str, attribute: str):
    module = import_module(module_name)
    if not hasattr(module, attribute):
        raise RuntimeError(
            f"Required v2 API {module_name}.{attribute} is unavailable. "
            "Update v2-refactor and rerun the tests before enabling this cell."
        )
    return getattr(module, attribute)


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if BRANCH != "v2-refactor":
    raise RuntimeError(f"Expected v2-refactor, found {BRANCH!r}. v1-final is read-only.")

print(f"Project: {PROJECT_ROOT}")
print(f"Branch:  {BRANCH}")


In [ ]:
from config import (
    ESM2_CACHE_DIR, ESM2_MODEL_NAME, ExperimentConfig, SPLITS_DIR,
    processed_pairs_path,
)

EXPERIMENT = ExperimentConfig(
    dataset="davis",
    split="random",
    model="advanced",
    seed=42,
    batch_size=16,
    epochs=30,
)
CACHE_PATH = ESM2_CACHE_DIR / f"{EXPERIMENT.dataset}_{ESM2_MODEL_NAME.split('/')[-1]}_residues.pt"
CACHE_METADATA_PATH = CACHE_PATH.with_suffix(".metadata.json")
ESM2_MODEL_REVISION = None  # set an immutable Hugging Face commit for final runs
MANIFEST_PATH = SPLITS_DIR / "manifests" / (
    f"{EXPERIMENT.dataset}_{EXPERIMENT.split}_seed{EXPERIMENT.seed}_manifest.csv"
)

RUN_EMBEDDING = False
RUN_MODEL_SMOKE_TEST = False
RUN_TRAINING = False

print(json.dumps(EXPERIMENT.to_dict(), indent=2))
print("ESM-2 cache:", CACHE_PATH)
print("ESM-2 metadata:", CACHE_METADATA_PATH)
print("Manifest:", MANIFEST_PATH)


## Steps

The cache maps preserved target IDs to detached residue tensors. Its required `.metadata.json` sidecar records model/sequence provenance and is created automatically by the builder. Keep the `.pt` file and sidecar together. ESM-2 parameters remain frozen; padding masks are retained separately and must be passed to the protein CNN and cross-attention block.


In [ ]:
if RUN_EMBEDDING:
    import pandas as pd

    build_cache = require_api(
        "src.preprocessing", "build_esm2_residue_embedding_cache"
    )
    pairs = pd.read_csv(processed_pairs_path(EXPERIMENT.dataset))
    residue_embedding_cache = build_cache(
        dataframe=pairs,
        output_path=CACHE_PATH,
        model_name=ESM2_MODEL_NAME,
        model_revision=ESM2_MODEL_REVISION,
    )
    if not CACHE_METADATA_PATH.exists():
        raise FileNotFoundError(
            f"Cache builder did not create required sidecar: {CACHE_METADATA_PATH}"
        )
    print({
        "cached_targets": len(residue_embedding_cache),
        "cache_path": str(CACHE_PATH),
        "metadata_path": str(CACHE_METADATA_PATH),
    })
else:
    print("ESM-2 cache generation skipped: RUN_EMBEDDING is False.")


In [ ]:
if RUN_MODEL_SMOKE_TEST:
    from src.models import create_advanced_dta_model

    model = create_advanced_dta_model(esm_embedding_dim=320)
    assert not any(
        "esm" in name.lower() for name, _ in model.named_parameters()
    )
    print(
        model.__class__.__name__,
        "contains no ESM module; training consumes detached cached residues."
    )
else:
    print("Model smoke test skipped: RUN_MODEL_SMOKE_TEST is False.")


In [ ]:
if RUN_TRAINING:
    if not CACHE_METADATA_PATH.exists():
        raise FileNotFoundError(
            f"Required cache metadata is missing: {CACHE_METADATA_PATH}"
        )
    load_cache = require_api(
        "src.preprocessing", "load_residue_embedding_cache"
    )
    run_advanced_experiment = require_api(
        "src.training", "run_advanced_experiment"
    )
    residue_embedding_cache = load_cache(CACHE_PATH)
    experiment_record = run_advanced_experiment(
        config=EXPERIMENT,
        residue_embedding_cache=residue_embedding_cache,
        split_manifest_path=MANIFEST_PATH,
    )
    print(json.dumps(experiment_record, indent=2, default=str))
else:
    print("Advanced training skipped: RUN_TRAINING is False.")


## Checks

Run the attention/model tests before training. Required invariants include `[B,N_atoms,128]` atom output, `[B,L,128]` residue output, six-column `edge_attr`, boolean valid-token masks, ignored padding, finite outputs, and no ESM module inside the trainable DTA model. ESM-2 is frozen only during cache generation; training consumes detached cached residue tensors. Pin `ESM2_MODEL_REVISION` to an immutable model commit for final provenance-verifiable caches.

## Next steps

Compare only matched dataset/split/seed/protocol records in notebook 08.
